# 05. Multi-Agent 선택 기준: 실패 분류에서 출발해 같은 조건으로 재기

Agent를 몇 개로 나눌지는 이 노트북의 질문이 아닙니다. **나누기 전에 무엇을 고쳤고, 나눈 뒤 무엇이 좋아졌으며, 그 대가가 얼마인지를 자기 실행 표로 말할 수 있는가**가 질문입니다. 이 노트북은 01~04에서 만든 구조를 같은 사례, 같은 도구, 같은 예산에 올려 재고, 마지막에 사례 유형별로 구조를 고른 이유를 기록으로 남깁니다.

이 노트북은 앞 노트북의 kernel 상태에 기대지 않고 단독으로 돕니다.

## 학습 목표

- 먼저 작업을 Workflow와 Agent로 나누고, Multi-Agent를 따질 대상이 모델이 맡은 칸뿐임을 확인합니다.
- 나누기 전에 Single의 실패를 분류하고, 프롬프트·도구·검색으로 고칠 문제를 먼저 고칩니다.
- 같은 사례에서 이득과 비용을 함께 재고, 조정 비용을 숫자로 말합니다.
- Persona 분리와 Context 분리를 실행으로 구분합니다.
- 코드, Skill, SubAgent 중 무엇을 쓸지 기준을 말합니다.
- 수치의 비교 조건을 읽고, 지연을 최장 경로로 분해합니다.
- Fast/Slow Path와 공통 게이트를 설계하고 오판의 경로를 확인합니다.
- 같은 조건의 비교표를 만들고 말할 수 있는 것과 없는 것을 구분합니다.
- 다섯 구조가 같은 하네스(예산, 도구 허용 목록, 응답 형식, 쓰기 게이트)를 지나기 때문에 구조만 다르게 비교된다는 점을 코드로 확인합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 0 | 먼저 작업을 Workflow와 Agent로 나눔 | 5 |
| 1 | 먼저 Single의 실패를 분류 | 8 |
| 2 | 이득과 비용 | 6 |
| 3 | Context 경계 vs Persona | 6 |
| 4 | Skill vs SubAgent | 6 |
| 5 | 수치 읽기와 지연 분해 | 5 |
| 6 | Fast/Slow Path | 6 |
| 7 | 같은 조건 비교표 | 12 |
| 8 | 선택 기록 | 3 |
| 9 | 직접 해보기와 정리 | 3 |

## 호출 규모

**이 과정에서 가장 비싼 노트북입니다.** 여러 구조를 같은 사례로 돌리기 때문입니다. 아래는 실행 전 추정이고, 각 유료 셀이 실행 직전에 규모를 한 줄로 알립니다. 수강생 본인의 크레딧을 씁니다.

| 범위 | 모델 호출 | EXA |
|---|---:|---:|
| 기본 실행 | 약 120~170회, 5~8분 | 0회 |
| `RUN_OPTIONAL = True` (첫 코드 셀, 4유형 10사례 × 5구조 × 1회) | 약 220~330회 더, 10~15분 | 0회 |
| 직접 해보기의 `RUN_EXERCISE = True` | 합쳐서 20~70회 더 | 0회 |

`RUN_OPTIONAL`과 `RUN_EXERCISE`는 기본값이 `False`라 전체 실행에 비용을 더하지 않습니다. 시간 여유와 크레딧을 보고 직접 켜세요.

In [ ]:
import inspect
import json
import re
import time
from collections import Counter, defaultdict
from uuid import uuid4

from deepagents import create_deep_agent
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.messages import ToolMessage

from mas_lab.contracts import FinalAnswer, Proposal, validate_final
from mas_lab.evaluation import evaluate_row, load_gold, summarize_rows
from mas_lab.harness import task_targets
from mas_lab.hybrid import FAST_TOOLS, build_hybrid
from mas_lab.ledger import check_refund
from mas_lab import runner
from mas_lab.runner import FIXED_BUDGET, STRUCTURES, build_retriever, make_desk, run_case, run_grid
from mas_lab.structures import OUTCOME_GUIDE, OWNER_TOOLS, build_handoff, build_single, build_supervisor, gate_answer, scope_line
from mas_lab.trace import ModelMeter, longest_path, timeline
from mas_lab.view import bars, message_table, read_design_log, show_table, write_design_log
from mas_lab.workers import ALL_TOOLS, DENY_WRITES, ROLE_TOOLS, SKILL_TOOLS, allowlist_middleware, subagent_spec
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import ROOT, announce, key_status, make_model, require_key

settings = require_key()  # 키가 없으면 여기서 멈추고 .env에 채우는 방법을 알려 줍니다.
print(key_status(settings))
model = make_model()  # 객체를 만들 뿐 호출이 아닙니다.
cases = load_cases()
case_by_id = {case["id"]: case for case in cases}
gold = load_gold()  # 사후 평가에만 씁니다. 실행 경로의 프롬프트와 도구로 들어가지 않습니다.
RUN_OPTIONAL = False  # True로 바꾸면 7절의 전체 비교(모델 약 220~330회, 10~15분)를 실행합니다.
print(f"사례 {len(cases)}개, 유형 {sorted({case['type'] for case in cases})}, RUN_OPTIONAL={RUN_OPTIONAL}")

## 0. 먼저 작업을 Workflow와 Agent로 나눕니다 (5분)

**문제.** 이 Part의 질문은 "Agent를 나눌까?"입니다. 그런데 이 질문은 작업을 먼저 Workflow와 Agent로 나타낸 뒤에야 뜻이 있습니다. 순서가 정해진 일, 규칙으로 판정할 수 있는 일, 사람이 승인해야 하는 일은 코드나 사람의 몫이고, 모델의 판단이 필요한 칸만 Agent의 몫입니다. 모델이 맡지 않은 칸을 여러 Agent로 나눈다고 판단이 더 좋아지지는 않습니다.

아래 셀은 03에서 만든 hybrid 그래프를 **만들기만 하고 부르지 않습니다.** 그래프에서 노드 이름을 읽고, 노드마다 누가 맡는지(코드, 모델, 사람)를 표로 적습니다. 담당은 그래프 안에서 자동으로 알 수 없으므로 셀 안의 사전에 사람이 적고, 그래프에 있는 노드가 사전에 없으면 멈춥니다.

**먼저 예측하세요.** 노드가 열 개 남짓인 이 그래프에서 모델이 맡은 노드는 몇 개일까요? 나머지는 코드와 사람 중 누구의 몫일까요? 표를 보기 전에 노드 이름으로 적어 보세요.

In [ ]:
# [C] 모델을 부르지 않습니다. 그래프를 만들기만 하고 부르지 않습니다.

# 1단계: 사례 하나의 Desk를 만들고 hybrid 그래프를 만듭니다. 만드는 일은 모델 호출이 아닙니다.
split_desk = make_desk(case_by_id["complex-double-pay"], label="section0")
split_graph = build_hybrid(split_desk, model)

# 2단계: 컴파일된 그래프에서 노드 이름을 읽습니다. 시작과 끝 표지는 일이 아니라서 뺍니다.
graph_nodes = [name for name in split_graph.get_graph().nodes if not name.startswith("__")]

# 3단계: 노드마다 누가 맡는지와 이유를 적습니다. 코드를 읽고 사람이 정한 값입니다.
NODE_OWNER = {
    "scope": ("코드", "질문에 다른 고객 이름이 있는지 문자열로 확인합니다. 판단이 필요 없습니다."),
    "router": ("모델", "fast와 slow 중 어느 경로가 맞는지는 질문의 뜻을 읽어야 정할 수 있습니다."),
    "fast": ("모델", "조사 한두 번으로 고객에게 보낼 답을 씁니다."),
    "worker": ("모델", "결제, 규정, 장애 중 하나를 도구로 조사하고 사실을 정리합니다."),
    "retry": ("모델", "시스템 오류로 실패한 조사를 다시 돌립니다. 다시 돌 때는 Worker가 모델을 씁니다."),
    "join": ("코드", "Worker 결과를 모으고 빠진 것이 있는지 규칙으로 셉니다."),
    "writer": ("모델", "Worker 결과를 합쳐 고객에게 보낼 답을 씁니다."),
    "validate": ("코드", "인용 소속, 금액 출처, outcome 일관성을 규칙으로 검사합니다."),
    "gate": ("코드", "환불 정책을 코드로 판정합니다. 모델은 환불을 직접 실행하지 못합니다."),
    "approve": ("사람", "승인이 필요한 환불은 사람의 응답을 기다립니다 (interrupt)."),
    "execute": ("코드", "승인된 환불만 멱등 키로 원장에 씁니다."),
}

# 4단계: 그래프에는 있는데 사전에 없는 노드가 있으면 조용히 넘어가지 않고 멈춥니다.
missing_nodes = [name for name in graph_nodes if name not in NODE_OWNER]
if missing_nodes:
    raise ValueError(f"담당을 적지 않은 노드가 있습니다: {missing_nodes}. NODE_OWNER에 담당과 이유를 더하세요.")

# 5단계: 그래프의 노드 순서대로 표를 만들고, 담당별 개수를 함께 셉니다.
split_rows = [{"노드": name, "담당(코드/모델/사람)": NODE_OWNER[name][0], "이유": NODE_OWNER[name][1]} for name in graph_nodes]
show_table(split_rows)
owner_counts = Counter(row["담당(코드/모델/사람)"] for row in split_rows)
print("담당별 노드 수:", dict(owner_counts))
print("Multi-Agent를 따질 노드(담당이 모델):", [row["노드"] for row in split_rows if row["담당(코드/모델/사람)"] == "모델"])
print("모델 호출 0회")

**관찰할 것.** 표의 `담당` 칸을 위에서 아래로 읽고, 예측과 비교하세요.

- Multi-Agent를 나눌지 따지는 대상은 `담당`이 **모델**인 행뿐입니다. `scope`, `join`, `validate`, `gate`, `execute`처럼 코드가 맡은 행을 Agent 여러 개로 쪼개도 새로 생기는 판단이 없으므로 얻는 것이 없고 조정 비용만 듭니다.
- `approve`처럼 사람이 맡은 행은 Agent가 아니라 승인 흐름의 문제입니다. 04에서 다룬 `interrupt`가 그 자리입니다.
- 모델이 맡은 행 안에서도 `router`와 `fast`는 한 번의 판단이고, `worker`는 서로 다른 조사입니다. 07절까지 이 노트북이 재는 것은 "모델이 맡은 칸을 몇 개의 Agent로 나눌 때 이득과 비용이 어떻게 달라지는가"입니다.
- 담당을 바꾸면 질문도 바뀝니다. 예를 들어 `validate`를 모델에게 맡기는 쪽으로 바꾸면 그 칸이 새로 Multi-Agent를 따질 대상이 되지만, 규칙으로 정할 수 있는 검사를 모델에게 넘긴 이유를 먼저 말해야 합니다.

## 1. 먼저 Single의 실패를 분류 (8분)

**문제.** "복잡하니까 Agent를 나누자"는 말은 대개 실패를 보기 전에 나옵니다. 그런데 Single이 틀린 이유가 검색어 하나, 도구 설명 한 줄이었다면 Agent를 셋으로 늘려도 같은 실패가 셋으로 복제될 뿐입니다. 나누기 전에 실패를 먼저 분류합니다.

아래 표는 **오류 통계가 아니라 진단 틀**입니다. "몇 %가 이 칸에 들어간다"고 말하는 자료가 아니라, 자기 실패를 어디서부터 점검할지 정하는 질문지입니다.

| 칸 | 어떻게 드러나는가 | 먼저 점검할 것 | 분리를 검토할 근거 |
|---|---|---|---|
| 정보가 부족함 | 필요한 원문이나 사실을 찾지 못하고 일반 지식으로 채우거나 비워 둠 | 검색어, 코퍼스, `not_found` 처리 지침 | 사실의 출처가 서로 달라 조사마다 다른 도구와 절차가 필요할 때 |
| 도구를 잘못 고름 | 엉뚱한 도구를 부르거나 필요한 도구를 건너뜀 | 도구 이름과 설명, 프롬프트의 지침 | 설명을 고쳐도 도구 수 때문에 계속 헷갈리고, 역할별 허용 목록이 실제로 선택지를 줄일 때 |
| Context가 너무 큼 | 호출을 거듭할수록 입력 토큰이 불어나고 앞의 조건을 놓침 | 도구 결과 크기, 필요한 원문만 읽기 | 줄여도 서로 무관한 조사의 원문이 한 Context에 남아 판단을 방해할 때 |
| 단계 의존성이 강함 | 앞 단계의 값 없이 뒤 단계를 수행하거나 순서를 어김 | 코드로 순서 고정(03의 순차 흐름) | 이 칸은 대개 **나눌 근거가 아닙니다.** 그래프로 순서를 고정하는 쪽이 먼저입니다 |

**먼저 예측하세요.** 아래 네 사례를 실행하기 전에, 각각 위 표의 어느 칸에 들어갈 실패가 날 것 같은지(또는 실패하지 않을지) 한 줄씩 적으세요.

- `complex-double-pay`: 결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.
- `insufficient-unknown-payment`: 결제 번호 P-7777도 환불해 주세요.
- `scope-beta-from-alpha`: 베타 계약 기준으로 크레딧 비율을 알려 주세요.
- `complex-ops-log`: 타임아웃 점검 순서와 운영 접근 로그의 보관 기간을 함께 묻는 질문입니다.

In [ ]:
announce("Single 기준선 4사례", model_calls=20)
BASELINE_IDS = ["complex-double-pay", "insufficient-unknown-payment", "scope-beta-from-alpha", "complex-ops-log"]
single_rows = [run_case("single", case_by_id[case_id], model=model) for case_id in BASELINE_IDS]
RUN_COLUMNS = ["structure", "case_id", "status", "outcome", "model_calls", "tool_calls", "input_tokens", "output_tokens", "elapsed_seconds"]
show_table(single_rows, columns=RUN_COLUMNS)

In [ ]:
single_checked = [evaluate_row(row, gold) for row in single_rows]
CHECK_COLUMNS = ["case_id", "자동 검사", "기대 outcome", "실제 outcome", "누락 사실", "금지 포함", "누락 문서", "실패 칸"]
show_table(single_checked, columns=CHECK_COLUMNS)
print("자동 검사는 문자열 포함 검사입니다. '정답'이 아니므로 실패 행의 답을 직접 읽습니다.")
for row, check in zip(single_rows, single_checked, strict=True):
    if not check["자동 검사"]:
        print(f"\n[{row['case_id']}] 누락 사실 {check['누락 사실']}\n{row['answer']}")

In [ ]:
def signals(row):
    """판결이 아니라 신호입니다. 어느 칸인지는 사람이 고릅니다."""
    counts = Counter(row["tools_called"])
    missing = [doc for doc in gold[row["case_id"]]["required_docs"] if doc not in row["retrieved_docs"]]
    inputs = [call["input_tokens"] for call in row["calls"] if call["input_tokens"] is not None]
    return {"사례": row["case_id"], "도구를 불렀는가": bool(counts), "부른 도구": dict(counts),
            "필요한 문서를 찾았는가": not missing, "찾지 못한 문서": missing,
            "같은 도구 최대 반복": max(counts.values(), default=0),
            "첫 호출 입력 토큰": inputs[0] if inputs else None, "마지막 호출 입력 토큰": inputs[-1] if inputs else None,
            "입력 토큰 증가(배)": round(inputs[-1] / inputs[0], 1) if inputs and inputs[0] else None}

show_table([signals(row) for row in single_rows])
print("모델 호출 0회. 이미 끝난 실행의 기록만 읽습니다.")

In [ ]:
# 위 두 표를 보고 직접 적으세요. 칸은 정보가 부족함, 도구를 잘못 고름, Context가 너무 큼, 단계 의존성이 강함, 자동 검사 문제, 실패 아님 중 하나입니다.
DIAGNOSIS = {case_id: {"칸": "미판정", "근거": "미판정"} for case_id in BASELINE_IDS}
# DIAGNOSIS["complex-double-pay"] = {"칸": "...", "근거": "어느 표의 어느 칸을 보고 그렇게 판단했는지"}

show_table([{"사례": case_id, **entry} for case_id, entry in DIAGNOSIS.items()])
print(f"미판정 {sum(entry['칸'] == '미판정' for entry in DIAGNOSIS.values())}건")

**관찰할 것.** 두 표를 나란히 읽으세요. 자동 검사 표의 `실패 칸`이 어느 검사에서 떨어졌는지를 말하고, 신호표는 그 이유의 단서를 줍니다.

- `실패 칸`에 `필수 사실`이 있으면 **답 전문부터 읽으세요.** 표 아래에 실패 행의 답을 출력해 두었습니다. 같은 사실을 다른 표현으로 썼다면 Agent의 실패가 아니라 자동 검사의 한계입니다. 이때는 구조를 바꿔도, 프롬프트를 고쳐도 고칠 대상이 아닙니다.
- 실제로 사실이 빠졌고 `필요한 문서를 찾았는가`가 False면 먼저 **정보가 부족함** 칸을 의심합니다. 찾았는데도 사실이 빠졌다면 원문을 읽고도 쓰지 않은 것이니 지침 쪽입니다.
- `부른 도구`에 필요한 도구가 없으면 **도구를 잘못 고름** 칸입니다. 예를 들어 보상을 물었는데 `lookup_availability`가 없는 경우입니다.
- `입력 토큰 증가(배)`가 크고 앞에서 확인한 조건을 답에서 놓쳤다면 **Context가 너무 큼** 칸을 봅니다. 증가만으로 실패라고 부르지 않습니다.
- `실패 칸`이 비어 있는 사례도 있을 수 있습니다. 그 사례는 Single로 충분하다는 뜻이 아니라, **이번 1회 관찰에서 자동 검사가 잡지 못했다**는 뜻입니다.

모든 사례가 통과했다면 실패 근거 없이 나누려는 것이니, 나눌 이유를 실패가 아닌 다른 곳(병렬 지연, 권한 분리)에서 찾아야 합니다. 어느 쪽이든 `DIAGNOSIS`에 근거 칸의 이름을 적으세요. 이제 작업을 나누기 전에 고칠 수 있는 칸을 하나 골라 고쳐 봅니다.

In [ ]:
def run_custom(label, case, build):
    """run_case와 같은 예산, 게이트, 검증을 쓰고 조립만 바꿉니다. 원문 메시지까지 돌려줍니다. 실패도 행으로 남깁니다."""
    run_id, desk, meter = uuid4().hex, make_desk(case, label=label), ModelMeter()
    row = {"run_id": run_id, "structure": label, "case_id": case["id"], "type": case["type"], "repeat": 1, "status": "ok", "outcome": None,
           "errors": [], "answer": "", "uncertainty": [], "evidence_refs": [], "raw_messages": []}
    started = time.perf_counter()
    try:
        state = build(desk).invoke({"messages": [{"role": "user", "content": case["question"]}]}, config={"callbacks": [meter], "recursion_limit": 60})
        row["raw_messages"], final = state["messages"], state.get("structured_response")
        if final is None:
            row.update(status="validation_failed", errors=["최종 응답이 없습니다."])
        else:
            final = gate_answer(final, desk, request_id=run_id, question=case["question"])
            problems = validate_final(final, desk.known_refs(), desk.source_text())
            row.update(status="validation_failed" if problems else "ok", errors=problems, outcome=final.outcome, answer=final.text,
                       evidence_refs=final.evidence_refs, uncertainty=final.uncertainty)
    except BudgetExceeded as error:
        row.update(status="budget_exceeded", errors=[str(error)])
    except Exception as error:  # 노트북을 멈추지 않고 실패를 기록합니다.
        row.update(status="model_error", errors=[f"{type(error).__name__}: {error}"[:300]])
    totals, report = meter.totals(), desk.budget.report()
    row.update(model_calls=totals["model_calls"], tool_calls=report["counts"].get("tool", 0), input_tokens=totals["input_tokens"],
               output_tokens=totals["output_tokens"], usage_missing=totals["usage_missing"], elapsed_seconds=round(time.perf_counter() - started, 3),
               ledger_refunds=len(desk.ledger.refunds), calls=meter.calls, spans=desk.trace.spans, tools_called=[e["name"] for e in report["events"] if e["event"] == "tool"],
               denied=[e["name"] for e in report["events"] if e["event"] == "tool_denied"], retrieved_docs=sorted({e.doc_id for e in desk.context.evidence.values()}),
               cited_docs=sorted({e.doc_id for ref, e in desk.context.evidence.items() if ref in row["evidence_refs"]}))
    return row

print("run_custom", inspect.signature(run_custom), "| 고정 예산:", FIXED_BUDGET, "| 모델 호출 0회")

In [ ]:
announce("프롬프트 한 줄만 고쳐 다시 실행", model_calls=8)
PATCHES = {  # 정답 값을 넣지 않습니다. 행동 지침만 한 줄 고칩니다.
    "complex-double-pay": "요청이 여러 개 섞이면 요청마다 필요한 조회를 모두 마친 뒤 답하세요. 보상을 물으면 가용률 조회와 SLA·개별 약정 검색을 모두 거치세요.",
    "insufficient-unknown-payment": "조회 결과가 not_found이면 다른 결제로 바꿔 답하지 말고 outcome을 insufficient로 두어 조회되지 않았다고 쓰세요.",
    "scope-beta-from-alpha": "현재 범위가 아닌 고객의 계약을 물으면 outcome을 out_of_scope로 두고 그 계약의 내용을 단정하지 마세요.",
    "complex-ops-log": "질문에 주제가 둘 이상이면 주제마다 따로 검색하고 evidence_refs에 주제별 evidence_id를 모두 넣으세요."}
failed_ids = [check["case_id"] for check in single_checked if not check["자동 검사"]]
target_id = failed_ids[0] if failed_ids else BASELINE_IDS[0]
PROMPT_PATCH = PATCHES[target_id]

def build_patched_single(desk):
    return create_deep_agent(model=model, tools=desk.tools("all"), name="single",
                             system_prompt=f"고객 요청을 처리하는 Agent입니다. {scope_line(desk)}\n{OUTCOME_GUIDE}\n{PROMPT_PATCH}",
                             middleware=allowlist_middleware(desk.budget, ALL_TOOLS), permissions=DENY_WRITES,
                             response_format=ProviderStrategy(FinalAnswer))

print("대상 사례:", target_id, "(자동 검사 실패 사례)" if failed_ids else "(실패 사례가 없어 첫 사례로 변동만 봅니다)")
patched_row = run_custom("single+한 줄", case_by_id[target_id], build_patched_single)
print("상태:", patched_row["status"], "모델", patched_row["model_calls"], "회")

In [ ]:
before_row = next(row for row in single_rows if row["case_id"] == target_id)
after_check, before_check = evaluate_row(patched_row, gold), evaluate_row(before_row, gold)
compare = [{**{key: row[key] for key in ("structure", "status", "outcome", "model_calls", "input_tokens", "elapsed_seconds")},
            "도구": dict(Counter(row["tools_called"])), "자동 검사": check["자동 검사"], "실패 칸": check["실패 칸"], "누락 사실": check["누락 사실"]}
           for row, check in ((before_row, before_check), (patched_row, after_check))]
show_table(compare)
print("바꾼 줄:", PROMPT_PATCH)
print(f"\n[고치기 전 답]\n{before_row['answer']}\n\n[고친 뒤 답]\n{patched_row['answer']}\n")
print("바꾼 곳은 이 한 줄뿐입니다. 도구, 모델, 예산, 사례는 그대로이며 양쪽 모두 1회 관찰입니다. 모델 호출 0회.")

**관찰할 것.** 표의 두 행에서 `자동 검사`, `실패 칸`, `도구`를 위아래로 비교하세요.

- `실패 칸`이 줄었다면 먼저 **두 답을 나란히 읽으세요.** 도구 호출과 내용이 달라져서 통과했다면 이 문제는 **분리가 필요 없었던 문제**일 가능성이 큽니다. 한 줄로 고칠 수 있는 것을 조직도로 고치면 비용만 늘어납니다. 내용은 같고 표현만 달라졌는데 통과했다면, 고쳐진 것은 Agent가 아니라 자동 검사에 걸린 문자열입니다.
- 그대로이거나 다른 칸이 새로 떨어졌다면 다음 절의 근거가 됩니다. 다만 먼저 `부른 도구`가 달라졌는지 보세요. 지침이 행동을 바꾸지 못한 것과, 행동은 바뀌었는데 결과가 그대로인 것은 다른 문제입니다.
- 셀 출력의 `대상 사례`가 "실패 사례가 없어 첫 사례로 변동만 봅니다"라면 이 표는 프롬프트의 효과가 아니라 실행 사이의 변동을 보여 줍니다. 고친 줄의 공으로 읽지 마세요.
- 어느 쪽이든 각 1회입니다. 모델 결과는 호출마다 달라지므로 한 번 나아진 것을 "고쳐졌다"고 단정하지 않습니다. 나아졌다고 말하려면 같은 수정으로 반복해 보아야 하고, 그것이 해보기 1입니다.

다음 절은 이 사례를 포함한 복합 요청을 Supervisor로 돌렸을 때 무엇을 얻고 무엇을 내는지 잽니다.

## 2. 이득과 비용 (6분)

**문제.** 구조를 바꾼 뒤 품질만 보거나 비용만 보면 판단이 틀어집니다. 품질이 나아졌어도 토큰과 시간이 함께 늘었을 수 있고, 품질이 같으면 늘어난 비용이 무엇을 산 것인지 답해야 합니다. Worker에게 내린 지시와 돌려받은 결과도 Supervisor의 입력이 되므로 **조정 비용**입니다. Single에는 이 비용이 없습니다.

위임 지시와 결과 전달의 크기를 재려면 원문 메시지가 필요합니다. 그래서 이 절은 `run_case`와 같은 예산, 게이트, 검증을 쓰면서 메시지를 함께 돌려주는 `run_custom`으로 `build_supervisor`를 실행합니다. 구조는 `run_case("supervisor", ...)`와 같습니다.

**먼저 예측하세요.** 같은 `complex-double-pay`를 Supervisor로 돌리면 입력 토큰은 Single의 몇 배가 될까요? 자동 검사는 나아질까요, 그대로일까요? 시간은 줄까요, 늘까요? 세 가지를 적고 넘어가세요.

In [ ]:
announce("Supervisor로 같은 사례", model_calls=20)
double_pay = case_by_id["complex-double-pay"]
supervisor_row = run_custom("supervisor", double_pay, lambda desk: build_supervisor(desk, model))
show_table([supervisor_row], columns=RUN_COLUMNS)
show_table(message_table(supervisor_row["raw_messages"]))

In [ ]:
def coordination_bytes(row):
    """task 호출의 인자가 위임 지시, task의 ToolMessage가 결과 전달입니다. 둘 다 Supervisor의 입력이 됩니다."""
    messages = row.get("raw_messages", [])
    orders = [call["args"] for message in messages for call in (getattr(message, "tool_calls", None) or []) if call["name"] == "task"]
    results = [str(message.content) for message in messages if isinstance(message, ToolMessage) and message.name == "task"]
    return {"위임 횟수": len(orders), "위임 지시 바이트": sum(len(json.dumps(order, ensure_ascii=False).encode()) for order in orders),
            "결과 전달 바이트": sum(len(text.encode()) for text in results)}

def side_by_side(row):
    check, first = evaluate_row(row, gold), next((call["input_tokens"] for call in row["calls"] if call["input_tokens"] is not None), None)
    return {"구조": row["structure"], "호출": row["model_calls"], "첫 호출 입력 토큰": first, "입력 토큰": row["input_tokens"], "출력 토큰": row["output_tokens"], "시간": row["elapsed_seconds"],
            "자동 검사": check["자동 검사"], "실패 칸": check["실패 칸"], "누락 사실": check["누락 사실"], **coordination_bytes(row)}

show_table([side_by_side(single_rows[0]), side_by_side(supervisor_row)])
print("Single은 위임이 없어 조정 칸이 0입니다. usage가 없는 호출은 입력 토큰에 더하지 않습니다. 모델 호출 0회.")

**관찰할 것.** 표의 `입력 토큰`, `시간`, `자동 검사`, `위임 지시 바이트`, `결과 전달 바이트`를 함께 읽으세요. 경우를 나누면 다음과 같습니다.

- **자동 검사가 Single 실패, Supervisor 통과**: 비용이 품질을 산 경우처럼 보입니다. 그렇게 말하기 전에 두 가지를 확인하세요. Single의 `누락 사실`이 정말 빠진 것인지 앞 절의 답 전문으로 확인합니다. 표현 차이였다면 품질 차이가 아닙니다. 그리고 앞 절에서 프롬프트 한 줄로도 고쳐졌다면 같은 품질을 더 싸게 얻을 수 있었던 것입니다.
- **둘 다 통과**: 품질이 같다면 Supervisor 쪽에서 더 든 비용은 무엇을 샀나요? 답할 수 있는 후보는 지연 단축, 역할별 도구 제한, 원문 격리입니다. 후보를 표의 칸으로 증명하지 못하면 비용만 는 것입니다. 늘어나는 칸은 `호출`, `시간`, 조정 바이트처럼 실행마다 다를 수 있으니 칸별로 따로 읽으세요. Single의 `입력 토큰`이 오히려 더 크게 나왔다면 Single이 이번에 같은 도구를 되풀이해 부른 것인지 앞 절 신호표의 `같은 도구 최대 반복`으로 확인하세요. 1회씩 잰 값이라 그 차이는 구조가 아니라 실행 변동일 수 있습니다.
- **Supervisor 실패**: `실패 칸`과 `누락 사실`을 보고, 위 메시지 표에서 Worker가 그 사실을 돌려줬는데 Supervisor가 놓쳤는지 찾으세요. 돌려받은 뒤 잃은 것은 전달 손실입니다.

`결과 전달 바이트`는 Worker 요약의 크기입니다. 이것이 작으면 Supervisor의 Context는 작지만 원문은 Worker에 남아 있어 나중에 다시 확인할 수 없습니다. 작을수록 좋다고 읽지 마세요. 그럼 Worker를 나누는 기준은 무엇일까요? 다음 절에서 역할 이름이 기준이 되는지 시험합니다.

## 3. Context 경계 vs Persona (6분)

**문제.** "결제 담당, 규정 담당, 장애 담당"이라는 이름을 붙이면 나눈 것처럼 보입니다. 그런데 이름이 다르다고 Worker가 보는 도구와 Context까지 달라지지는 않습니다. 앞 절의 Worker는 `ROLE_TOOLS`로 도구 범위가 서로 다른 Context 분리였습니다. 이번에는 **같은 프롬프트와 이름을 유지한 채** 도구만 모두 열어 Persona만 남긴 Worker를 돌립니다.

역할별 사용량은 Worker의 middleware에 칸을 하나 더해 잽니다. 도구 범위 말고는 Supervisor의 프롬프트, 사례, 예산을 그대로 둡니다.

02 3절에서 비교한 프레임워크가 자동으로 붙이는 `general-purpose` Worker도 같은 이야기입니다. 이 Worker는 부모와 같은 것을 보므로 Context 경계가 없고, 이름만 다른 Persona와 다르지 않습니다.

**먼저 예측하세요.** 이름만 다른 Worker 셋에게 모든 도구를 주면 어떻게 될까요? 각 Worker가 자기 도구만 쓸까요? 역할 밖 도구 호출은 몇 번일까요? 입력 토큰은 Context 분리 때보다 늘까요? 숫자로 적어 보세요.

In [ ]:
# 실패 재현: 역할 이름만 다르고 도구는 모두 같은 Worker 셋을 실행합니다.
announce("Persona만 나눈 Worker", model_calls=20)
role_log = defaultdict(lambda: {"모델 호출": 0, "입력 토큰": 0, "도구": []})

def role_meter(role):
    @wrap_model_call
    def metered_model(request, handler):
        response = handler(request)
        usage = next((m.usage_metadata for m in response.result if getattr(m, "usage_metadata", None)), None) or {}
        role_log[role]["모델 호출"] += 1
        role_log[role]["입력 토큰"] += usage.get("input_tokens", 0)
        return response

    @wrap_tool_call
    def metered_tool(request, handler):
        role_log[role]["도구"].append(request.tool_call["name"])
        return handler(request)
    return [metered_model, metered_tool]

def build_persona(desk):
    specs = [{**subagent_spec(role, desk, model), "tools": desk.tools("all"),
              "middleware": [*allowlist_middleware(desk.budget, ALL_TOOLS), *role_meter(role)]} for role in ("billing", "policy", "outage")]
    prompt = (f"고객 요청을 조사 Worker에게 task로 위임하고 결과로 최종 응답합니다. {scope_line(desk)} 서로 독립인 조사는 한 메시지에서 동시에 위임하세요.\n"
              f"금액 계산은 calculate_credit을 쓰세요.\n{OUTCOME_GUIDE}")
    return create_deep_agent(model=model, tools=desk.tools("writer"), subagents=specs, system_prompt=prompt, permissions=DENY_WRITES, name="supervisor-persona",
                             middleware=allowlist_middleware(desk.budget, frozenset({"task"}) | ROLE_TOOLS["writer"]), response_format=ProviderStrategy(FinalAnswer))

persona_row = run_custom("persona", double_pay, build_persona)
show_table([side_by_side(supervisor_row), side_by_side(persona_row)])

In [ ]:
def role_of_call(call):
    """모델 호출 때 보인 도구로 역할을 읽습니다. 모든 도구가 보이면 역할을 구분할 수 없습니다."""
    tools = set(call.get("tools") or [])
    if "task" in tools:
        return "supervisor"
    if not tools:
        return "router"
    matched = [role for role, names in ROLE_TOOLS.items() if tools <= names]
    return matched[0] if matched else ("fast" if tools == ROLE_TOOLS["policy"] | ROLE_TOOLS["billing"] else "전체 도구")

def split_roles(row):
    table = defaultdict(lambda: {"모델 호출": 0, "입력 토큰": 0, "도구": []})
    for call in row["calls"]:
        table[role_of_call(call)]["모델 호출"] += 1
        table[role_of_call(call)]["입력 토큰"] += call["input_tokens"] or 0
    for name in row["tools_called"]:
        owner = next((role for role, names in ROLE_TOOLS.items() if name in names), "supervisor")
        table["supervisor" if name in ("task", "calculate_credit") else owner]["도구"].append(name)
    return table

def role_rows(label, table):
    return [{"구조": label, "역할": role, **{k: entry[k] for k in ("모델 호출", "입력 토큰")}, "호출한 도구": dict(Counter(entry["도구"])),
             "역할 밖 도구 호출": sum(name not in ROLE_TOOLS.get(role, frozenset(entry["도구"])) for name in entry["도구"])} for role, entry in table.items()]

worker_tools = Counter(name for entry in role_log.values() for name in entry["도구"])
persona_table = {**role_log, "supervisor": {**split_roles(persona_row)["supervisor"], "도구": list((Counter(persona_row["tools_called"]) - worker_tools).elements())}}
show_table(role_rows("Context 분리", split_roles(supervisor_row)) + role_rows("Persona", persona_table))
print("거부된 도구 호출 Context 분리:", supervisor_row["denied"], "/ Persona:", persona_row["denied"], "| 모델 호출 0회")

In [ ]:
desk = make_desk(double_pay, label="shared")
specs = {role: subagent_spec(role, desk, model) for role in ("billing", "outage")}

def reached_desk(spec):
    """도구 함수가 잡고 있는 Desk를 꺼냅니다. 허용 목록과 무관하게 같은 프로세스의 같은 객체입니다."""
    return inspect.getclosurevars(spec["tools"][0].func).nonlocals["desk"]

show_table([{"Worker": role, "허용 도구": sorted(tool.name for tool in spec["tools"]), "같은 Desk": reached_desk(spec) is desk,
             "같은 원장": reached_desk(spec).ledger is desk.ledger} for role, spec in specs.items()])
reached_desk(specs["billing"]).ledger.refund("P-DEMO", 1, idempotency_key="shared-demo")
print("billing Worker 쪽에서 원장에 직접 쓴 뒤 outage Worker 쪽에서 본 원장 건수:", len(reached_desk(specs["outage"]).ledger.refunds))
print("이 실행의 임시 Desk에만 쓴 것입니다. 모델 호출 0회.")

**관찰할 것.** 역할 이름이 줄이는 것과 도구 범위가 줄이는 것을 구분합니다.

- 역할 표에서 `Persona` 행의 `호출한 도구`와 `역할 밖 도구 호출`을 보세요. 0이라면 프롬프트와 위임 지시만으로 모델이 **이번에는** 자기 몫만 한 것입니다. 지켜졌다고 해서 보장된 것은 아닙니다. 다음 호출에서 어길 수 있고, 어겨도 막는 코드가 없습니다.
- 1 이상이면 역할 이름이 경계가 아니었다는 직접 증거입니다. 어떤 도구를 왜 불렀는지 함께 찾아보세요.
- `입력 토큰`을 역할별로 비교하세요. 도구 스키마가 늘면 Worker의 호출마다 입력이 함께 늡니다. 다만 `모델 호출`이 다른 행은 합계로 비교하지 마세요. 원문을 한 번 더 읽은 Worker(예: `read_evidence`를 부른 `policy`)는 도구 수와 상관없이 합계가 큽니다. 줄어드는 비용은 이름이 아니라 **보이는 도구와 넘기는 Context**를 줄여서 생깁니다.
- 마지막 셀의 `같은 Desk`, `같은 원장`이 True라는 것은 분리된 Worker도 같은 프로세스의 같은 객체에 닿을 수 있다는 뜻입니다. **Context 분리는 보안 격리가 아닙니다.** 모델에게 쓰기 도구를 주지 않은 것이 제한일 뿐, 코드 수준의 경계는 없습니다. 실제 격리가 필요하면 프로세스, 자격 증명, 원장 쪽의 권한으로 별도로 설계합니다. 이 과정에서 원장에 닿는 경로를 `gate_answer` 하나로 모은 이유도 여기 있습니다.

정리하면 나누는 기준은 "누구처럼 말하는가"가 아니라 "무엇을 보고, 무엇을 쓸 수 있는가"입니다. 그럼 절차를 재사용하고 싶을 뿐이라면 Worker까지 나눌 필요가 있을까요?

## 4. Skill vs SubAgent (6분)

**문제.** Worker를 만드는 이유가 "환불 절차를 매번 설명하기 싫어서"라면, 별도 실행 Context까지 필요하지 않을 수 있습니다. 절차는 Skill로 불러올 수 있고, 불러온 절차는 같은 Context 안에서 실행됩니다. SubAgent는 별도 실행 Context로 위임합니다. 두 가지는 해결하는 문제가 다릅니다.

Skill은 메타데이터(이름과 설명)가 항상 보이고, 본문은 Agent가 필요할 때 읽습니다. 그래서 메타데이터 크기는 모든 호출의 비용이고, 본문 크기는 읽었을 때의 비용입니다. 먼저 실제 크기를 봅니다.

02 3절의 자동으로 붙는 `general-purpose` Worker는 부모의 도구를 그대로 받으므로, 별도 실행 Context로 위임할 이유가 보이지 않는 SubAgent의 예입니다.

**먼저 예측하세요.** `single_skills`는 Single보다 모델 호출이 늘까요? 절차를 읽는 호출이 더해지는 대신 도구를 헤매는 호출이 줄 수도 있습니다. 어느 쪽이 클지, 자동 검사는 달라질지 적으세요.

In [ ]:
skill_rows = []
for path in sorted((ROOT / "skills").glob("*/SKILL.md")):
    _, front, body = path.read_text(encoding="utf-8").split("---", 2)
    skill_rows.append({"Skill": path.parent.name, "메타데이터 바이트": len(front.encode()), "본문 바이트": len(body.encode()),
                       "description 앞부분": next(line for line in front.splitlines() if line.startswith("description:"))[13:60]})
show_table(skill_rows)
print("메타데이터는 항상 입력에 들어가고, 본문은 읽을 때만 들어갑니다. 모델 호출 0회.")

In [ ]:
announce("single_skills로 같은 사례", model_calls=12)
skills_row = run_case("single_skills", double_pay, model=model)
show_table([skills_row], columns=RUN_COLUMNS)
print("읽은 Skill:", skills_row["loaded_skills"] or "없음")
print("부른 도구:", dict(Counter(skills_row["tools_called"])))

In [ ]:
show_table([side_by_side(row) for row in (single_rows[0], skills_row, supervisor_row)])
show_table([
    {"기준": "절차 재사용", "Skill": "적합. 본문을 필요할 때 읽습니다", "SubAgent": "가능하지만 과합니다", "코드": "판단이 없는 절차는 코드가 낫습니다"},
    {"기준": "Context 분리", "Skill": "없음. 읽은 본문과 조회 결과가 같은 Context에 쌓입니다", "SubAgent": "있음. 부모는 마지막 메시지만 받습니다", "코드": "해당 없음"},
    {"기준": "도구·권한 분리", "Skill": "없음. 같은 Agent의 도구를 씁니다", "SubAgent": "허용 목록을 Worker마다 겁니다", "코드": "게이트처럼 강제할 수 있습니다"},
    {"기준": "병렬 필요", "Skill": "없음", "SubAgent": "한 메시지의 task 여러 개로 가능합니다", "코드": "Send로 코드가 정합니다"}])
print("판단표는 기준을 정리한 것입니다. 위 표의 수치가 어느 칸을 뒷받침하는지는 직접 대조하세요. 모델 호출 0회.")

**관찰할 것.** `읽은 Skill`이 비어 있으면 절차를 읽지 않고 답한 것입니다. 이 경우 Single과의 차이를 Skill의 효과로 말할 수 없습니다. 읽었다면 세 행의 `호출`, `입력 토큰`, `자동 검사`를 비교해 절차를 불러오는 비용이 무엇을 바꿨는지 보세요.

`첫 호출 입력 토큰`의 차이는 아직 아무 Skill도 읽기 전의 비용입니다. 위 표의 `메타데이터 바이트`만으로 설명되는지 따져 보세요. 메타데이터보다 차이가 훨씬 크다면 파일 도구 설명과 Skill 안내문까지 매 호출 함께 실린 것입니다. 또 `입력 토큰`은 호출마다의 입력을 더한 값이라서, 앞에서 읽은 내용이 뒤의 모든 호출에 반복됩니다. 그래서 본문이 작아도 합계는 크게 늘 수 있습니다. "Skill은 필요할 때만 읽으니 싸다"는 말도 이 표로 확인한 뒤에 하세요.

Skill과 SubAgent는 양자택일이 아닙니다. Worker가 Skill을 읽을 수도 있습니다. 판단 기준은 다음과 같이 정리할 수 있습니다.

- 같은 절차가 반복되지만 조회한 사실을 한 Context에서 함께 봐야 한다면 Skill을 먼저 검토합니다.
- 원문이 커서 부모 Context에서 떼어 놓아야 하거나 Worker마다 도구를 달리해야 한다면 SubAgent를 검토합니다.
- 판단 없이 입력을 곱하는 일이나 정책 검증은 코드에 둡니다. `calculate_credit`과 `check_refund`가 그 예입니다.

한 가지 전이 질문을 생각해 보세요. (`Day03/Day03_01.png`의 투자 심사 판서에서 가져온 질문입니다.) 투자 심사 보고서를 쓰는 Agent를 설계한다면 **문체, 데이터를 바라보는 관점, 심사역의 암묵지**는 어디에 담아야 할까요? 문체와 체크리스트는 Skill에 적을 수 있지만, 말로 꺼내 본 적이 없는 판단은 Skill에 적을 수 없습니다. 어떤 부분은 코드, 어떤 부분은 Skill, 어떤 부분은 사람의 검토로 남는지 자기 말로 나눠 보세요.

## 5. 수치 읽기와 지연 분해 (5분)

**문제.** "토큰이 3배"나 "2배 빠르다"같은 문장은 비교 조건 없이 돌아다니면 아무 뜻도 없습니다. 무엇과 무엇을 비교했는지(비교 대상), 어떻게 쟀는지(평가 방법), 몇 번 쟀는지(표본), 누가 쟀는지(측정자) 중 하나라도 비어 있으면 주장할 수 없습니다. 아래 셀은 **이 노트북에서 내가 방금 만든 표에서만** 문장을 뽑아 칸으로 나눕니다. 외부 수치는 넣지 않습니다.

지연도 마찬가지입니다. Worker를 셋으로 나누면 세 배 빨라진다는 말은 틀렸습니다. 지연은 Agent 수가 아니라 **가장 긴 실행 경로**로 정해지고, 그 경로에는 조정과 집계 시간이 포함됩니다. `supervisor` 행의 `spans`에는 Worker 구간이 남지만 Supervisor 자신의 조정 구간은 없습니다. 둘을 함께 보려고 여기서는 모델 호출 기록(`calls`)의 시작과 끝으로 구간을 만듭니다.

In [ ]:
def times(a, b):
    return round(a / b, 2) if a and b else "값 없음"

single_double = single_rows[0]
mean_single = sum(row["elapsed_seconds"] for row in single_rows) / len(single_rows)
BLANK_FIELDS = ["비교 대상", "평가 방법", "표본", "측정자"]
claims = [
    {"문장": f"Supervisor는 Single의 {times(supervisor_row['input_tokens'], single_double['input_tokens'])}배 입력 토큰을 썼다", "비교 대상": "complex-double-pay에서 single과 supervisor", "평가 방법": "ModelMeter 입력 토큰 합계", "표본": "각 1회", "측정자": "내 노트북 실행"},
    {"문장": f"Single+Skills는 Single의 {times(skills_row['elapsed_seconds'], single_double['elapsed_seconds'])}배 시간이 걸렸다", "비교 대상": "complex-double-pay에서 single과 single_skills", "평가 방법": "run_case의 elapsed_seconds", "표본": "각 1회", "측정자": "내 노트북 실행"},
    {"문장": f"Supervisor({supervisor_row['elapsed_seconds']:.0f}초)와 Single 평균({mean_single:.0f}초)을 비교하면 시간이 {times(supervisor_row['elapsed_seconds'], mean_single)}배다", "비교 대상": "", "평가 방법": "elapsed_seconds", "표본": "1회와 4사례 평균", "측정자": "내 노트북 실행"},
    {"문장": "Supervisor가 Single보다 답의 품질이 좋다", "비교 대상": "complex-double-pay에서 single과 supervisor", "평가 방법": "", "표본": "각 1회", "측정자": ""}]

def verdict(claim):
    blanks = [field for field in BLANK_FIELDS if not claim[field]]
    if blanks:
        return f"주장할 수 없음: {', '.join(blanks)} 빈 칸"
    return "1회 관찰로만 말할 수 있음" if "1회" in claim["표본"] else "조건을 확인한 뒤 말할 수 있음"

show_table([{**claim, "판정": verdict(claim)} for claim in claims])
print("셋째 문장은 사례 수가 달라 비교 대상이 성립하지 않으므로 칸을 비웠습니다. 수치는 모두 이 노트북의 실행에서 왔습니다. 모델 호출 0회.")

In [ ]:
def call_spans(row):
    """모델 호출 하나를 구간 하나로 만듭니다. 호출 사이의 시간이 도구와 코드의 몫입니다."""
    done = [call for call in row["calls"] if call.get("end")]
    origin = min((call["start"] for call in done), default=0.0)
    return [{"actor": f"{role_of_call(call)}#{index}", "kind": role_of_call(call), "start": round(call["start"] - origin, 3), "end": round(call["end"] - origin, 3)}
            for index, call in enumerate(done)]

spans = call_spans(supervisor_row)
durations = {span["actor"]: span["end"] - span["start"] for span in spans}
depends = {span["actor"]: [other["actor"] for other in spans if other["start"] < span["start"] and other["end"] <= span["start"]] for span in spans}
serial = sum(durations.values())
critical = longest_path(durations, depends)
by_kind = defaultdict(float)
for span in spans:
    by_kind["Supervisor 조정·집계" if span["kind"] == "supervisor" else "Worker 합계"] += span["end"] - span["start"]
print(timeline(spans))
print()
print(bars({"실제 경과(ms)": supervisor_row["elapsed_seconds"] * 1000, "모델 직렬 합(ms)": serial * 1000, "모델 최장 경로(ms)": critical * 1000,
            **{f"{name}(ms)": seconds * 1000 for name, seconds in by_kind.items()}}))
print(f"\nWorker 역할 {sum(kind in ROLE_TOOLS for kind in {span['kind'] for span in spans})}개, 직렬 합 / 최장 경로 = {times(serial, critical)}배, 실제 경과 - 최장 경로 = 도구와 코드 {supervisor_row['elapsed_seconds'] - critical:.2f}초. 모델 호출 0회.")

**관찰할 것.** 첫 표에서 `판정` 칸을 읽으세요. 숫자가 맞아도 빈 칸이 있으면 주장이 되지 않습니다. 셋째 문장은 계산은 되지만 비교 대상이 다릅니다. 넷째 문장은 숫자조차 없고 측정자도 없습니다. 자기가 하고 싶은 말을 문장으로 추가해 같은 칸에 넣어 보세요. 다른 곳에서 본 수치도 이 네 칸으로 나누어 빈 칸을 확인할 수 있습니다.

둘째 출력은 Agent 수가 아니라 가장 긴 경로로 읽습니다.

- 타임라인에서 세로로 겹치는 막대가 동시 실행입니다. 겹치지 않으면 `Send`가 아니라 모델의 선택에 맡긴 병렬이 이번에는 일어나지 않은 것입니다.
- `모델 직렬 합`과 `모델 최장 경로`의 차이가 병렬로 줄어든 시간의 상한입니다. 비율이 Worker 수에 못 미치면 그 이유를 찾으세요. 대개 가장 느린 Worker와 Supervisor 호출이 경로에 남습니다.
- `Supervisor 조정·집계`는 나누면서 새로 생긴 시간입니다. 이 시간이 Worker 절약분보다 크면 나누기 전보다 느려집니다. Single 행의 시간과 같이 보세요.

그럼 모든 요청이 이 긴 경로를 지나야 할까요? 다음 절은 단순한 요청을 짧은 경로로 보내는 Fast Path를 다룹니다.

## 6. Fast/Slow Path (6분)

**문제.** 단순한 질문에 Worker 셋을 돌리는 것은 낭비입니다. 그래서 Router로 경로를 나눕니다. 그런데 Fast Path는 **안전 검사를 생략하는 지름길이 아닙니다.** 건너뛰는 것은 조사의 폭이지 검증이 아닙니다. 그리고 Router는 틀립니다. 복합 요청이 Fast로 가면 어떻게 되는지까지 봐야 Fast Path의 이득을 말할 수 있습니다.

`hybrid`는 범위 확인 → Router → Fast 또는 Worker 병렬 → 합성 → 검증 → 게이트 순서이며, Fast와 Slow 모두 같은 검증과 게이트를 지납니다. Fast에서 outcome이 근거 부족이거나 검증 문제가 있으면 Slow로 전환합니다.

**먼저 예측하세요.** `complex-double-pay`를 Fast Path로 강제하면 무슨 일이 일어날까요? Fast에는 가용률 조회와 크레딧 계산 도구가 없습니다. 검증이 실패해 Slow로 전환될까요, 아니면 보상 부분이 빠진 답이 검증을 통과할까요? `insufficient-unknown-payment`는 어떨까요? 두 사례를 각각 적으세요.

In [ ]:
announce("hybrid 단순 사례와 복합 사례", model_calls=30)
hybrid_rows = [run_case("hybrid", case_by_id[case_id], model=model) for case_id in ("simple-deadline", "complex-double-pay")]

def path_row(row, note=""):
    decided = [event.get("choice") for event in row["events"] if event["kind"] == "decision"]
    check, roles = evaluate_row(row, gold), Counter(role_of_call(call) for call in row["calls"])
    return {"사례": row["case_id"], "조건": note or "Router 판단", "경로 판단": decided, "Slow 전환": any(event["kind"] == "fallback_to_slow" for event in row["events"]),
            "호출": row["model_calls"], "역할별 호출": dict(roles), "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"], "상태": row["status"],
            "outcome": row["outcome"], "자동 검사": check["자동 검사"], "누락 사실": check["누락 사실"]}

show_table([path_row(row) for row in hybrid_rows])
print(timeline(hybrid_rows[1]["spans"]))

In [ ]:
# 실패 재현: Router의 판단을 무시하고 두 요청을 Fast Path로 강제합니다.
announce("Fast Path 강제 2사례", model_calls=24)
forced_rows = [run_case("hybrid", case_by_id[case_id], model=model, force_path="fast") for case_id in ("complex-double-pay", "insufficient-unknown-payment")]
show_table([path_row(row, "Fast 강제") for row in forced_rows] + [path_row(hybrid_rows[1])])
for row in forced_rows:
    print(row["case_id"], "사건 순서:", [event["kind"] for event in row["events"] if event["kind"] != "tool"])
    print("  답 앞부분:", row["answer"][:120].replace("\n", " "))

In [ ]:
beta_desk, alpha_desk = make_desk(case_by_id["approval-beta-small"], label="gate"), make_desk(double_pay, label="gate")
requests = [(alpha_desk, "P-8101", 1_000_000), (alpha_desk, "P-7002", 1_000_000), (alpha_desk, "P-9002", 1_200_000), (beta_desk, "P-5002", 300_000)]
gate_rows = []
for desk, payment_id, amount in requests:
    proposal = Proposal(action="refund", payment_id=payment_id, amount=amount, reason="Fast Path에서 온 쓰기 제안")
    answer = FinalAnswer(outcome="executed", text="환불했습니다.", proposal=proposal)  # 모델이 실행했다고 우겨도 게이트가 정합니다.
    gated = gate_answer(answer, desk, request_id="fast-demo")
    gate_rows.append({"고객": desk.scope.entity, "결제": payment_id, "제안의 outcome": answer.outcome, "게이트 판단": check_refund(proposal, desk.payments(), desk.scope.as_of).decision,
                      "최종 outcome": gated.outcome, "원장 건수": len(desk.ledger.refunds)})
show_table(gate_rows)
edges = build_hybrid(alpha_desk, model).get_graph().edges
show_table([{"출발": e.source, "도착": e.target, "조건부": e.conditional} for e in edges if e.source in ("fast", "writer", "validate")])
print("Fast와 writer는 모두 validate로 가고 게이트를 지납니다. 일반 함수 경로입니다. 모델 호출 0회.")

**관찰할 것.** Fast Path의 이득은 복구 경로의 비용까지 합쳐야 말할 수 있습니다. 표의 `Slow 전환`, `호출`, `입력 토큰`, `자동 검사`, `누락 사실`을 함께 읽으세요.

- `Slow 전환`이 True면 Fast에서 쓴 호출과 Slow에서 다시 쓴 호출을 **모두** 냈습니다. `역할별 호출`의 `fast` 칸이 복구 전에 버린 호출이고, 이 행에서 그만큼이 처음부터 Slow로 갔을 때보다 더 든 값입니다. Fast 오판이 잦을수록 평균 이득은 사라집니다.
- `Slow 전환`이 False인데 `누락 사실`이 있으면 더 위험한 경우입니다. 구조 검증은 인용 소속과 금액의 출처를 볼 뿐, **빠진 범위**는 보지 못합니다. 호출과 토큰은 적고 답은 그럴듯하므로, 비용 칸만 보면 이 행이 가장 좋아 보입니다. 단 `누락 사실`이 날짜 하나뿐이면(예: `9월20일`) 답 전문에 `2026-09-20`처럼 다른 표기로 들어 있는지 먼저 보세요. 표기 차이라면 이 위험이 아니라 자동 검사의 한계입니다.
- 어느 쪽이든 전환이 일어난 이유는 `사건 순서`의 `validate`, `fallback_to_slow`로 확인합니다. 전환 조건이 정말 필요한 경우를 다 잡는지 생각해 보세요. `gate`가 두 번 보이는 행은 승인 대기로 멈춘 경우입니다. 그래프 상태에 게이트 결과가 없어 실행기가 같은 요청 ID로 한 번 더 부르며, 같은 요청 ID라 원장에 두 번 쓰지 않습니다. 전환 뒤 `worker_checked`가 Worker마다 하나씩 보이면 합성 전에 코드가 각 Worker 결과를 거른 기록입니다. 사건의 `dropped`는 등록된 번호를 찾지 못해 뺀 근거 참조이고, `recovered`는 번호에 설명이 덧붙은 참조에서 등록된 번호만 가려 남긴 수입니다. 등록된 번호는 설명이 붙어 있어도 버리지 않고 남깁니다. `filled`는 사실은 적혔는데 거른 뒤 근거 참조가 하나도 남지 않은 결과에 그 Worker 자신의 도구 결과에서 채워 넣은 근거 참조의 수입니다. 칸의 값은 행의 `events`에서 읽습니다. `gate_skipped`는 코드가 질문의 결제 번호를 조회된 결제와 대조할 수 없었다는 흔적입니다(조회되지 않는 번호이거나 번호가 없을 때, 환불을 묻는 질문이거나 답이 거절·심사 결론일 때). 답 본문은 바뀌지 않고, 쓰기 제안이 없으니 원장도 그대로입니다.

게이트 표에서는 `제안의 outcome`이 모두 executed인데 `최종 outcome`과 `원장 건수`는 `게이트 판단`을 따릅니다. `allow`인 행만 executed와 1건이 되고, 나머지는 거절되거나 승인 대기와 심사로 멈춰 원장이 0건입니다. 쓰기 요청이 어느 경로로 오든 `check_refund`를 지납니다. 간선 표에서 `fast`와 `writer`가 같은 `validate`로 모이는 것이 그 구조적 근거입니다. 이제 지금까지 따로 본 구조들을 같은 조건의 표 하나로 모읍니다.

## 7. 같은 조건 비교표 (12분)

**문제.** 앞 절들의 수치는 사례와 조건이 조금씩 달라 구조를 고르는 근거로 쓸 수 없습니다. 비교표는 **바꾸는 변수를 하나로 줄인 표**입니다. 여기서 바꾸는 것은 구조뿐이고 모델, effort, 도구, 코퍼스, 예산, 기준일은 고정합니다. 어느 구조든 같은 `gate_answer`를 지납니다.

기본 구성은 표본이 작습니다. 그래서 이 표의 목적은 승자를 뽑는 것이 아니라, 무엇을 재면 말할 수 있는지 아는 것입니다.

**먼저 예측하세요.** 실행 전에 유형(simple, complex)마다 어느 구조의 입력 토큰이 가장 많을지, 자동 검사가 떨어지는 조합이 있다면 어디일지 적으세요.

**하네스가 일관성을 만듭니다.** 다섯 구조(`single`, `single_skills`, `supervisor`, `handoff`, `hybrid`)는 조립만 다르고, 같은 하네스를 지납니다. 같은 `Desk`의 예산 객체, 도구 허용 목록(`allowlist_middleware`), 응답 형식(`FinalAnswer`), 쓰기 게이트(`gate_answer`)가 그것입니다. 그래서 표의 차이를 구조 탓으로 읽을 수 있습니다. 하네스가 구조마다 달랐다면 차이가 구조 때문인지 하네스 때문인지 가를 수 없습니다.

프레임워크가 자동으로 붙이는 `general-purpose` Worker는 이 하네스 밖에서 돕니다 (02 3절). 이 과정은 `mas_lab`을 import할 때 그 Worker를 끄므로, 다섯 구조 어디에도 없어야 합니다. 아래 셀이 다섯 구조를 만들어(모델은 부르지 않습니다) 코드 객체와 소스에서 이를 읽습니다. `handoff`와 `hybrid`는 그래프이고 Agent가 노드 안에서 실행 때 만들어지므로, 그래프에 `task` 도구가 없습니다. 이 두 구조는 같은 설정으로 만든 Agent로 대신 확인합니다.

In [ ]:
# [C] 모델을 부르지 않습니다. 구조를 만들고 코드 객체와 소스를 읽기만 합니다.

# 1단계: 구조 이름마다 만드는 함수와, 소스를 읽을 함수를 짝지어 둡니다.
BUILDERS = {
    "single": lambda desk: build_single(desk, model),
    "single_skills": lambda desk: build_single(desk, model, skills=True),
    "supervisor": lambda desk: build_supervisor(desk, model),
    "handoff": lambda desk: build_handoff(desk, model),
    "hybrid": lambda desk: build_hybrid(desk, model),
}
BUILDER_SOURCE = {"single": build_single, "single_skills": build_single, "supervisor": build_supervisor,
                  "handoff": build_handoff, "hybrid": build_hybrid}
assert set(BUILDERS) == set(STRUCTURES), "runner.STRUCTURES와 이 셀의 구조 목록이 다릅니다."

# 2단계: 구조마다 허용 목록이 놓인 곳을 적습니다. 값은 코드 상수에서 그대로 가져옵니다.
ALLOWED = {
    "single": ALL_TOOLS,
    "single_skills": ALL_TOOLS | SKILL_TOOLS,
    "supervisor": {"Supervisor": frozenset({"task"}) | ROLE_TOOLS["writer"], **ROLE_TOOLS},
    "handoff": OWNER_TOOLS,
    "hybrid": {"fast": FAST_TOOLS, **ROLE_TOOLS},
}
GRAPH_STRUCTURES = {"handoff", "hybrid"}  # Agent가 노드 안에서 실행 때 만들어지는 구조

def allow_text(allowed):
    if isinstance(allowed, dict):
        return "; ".join(f"{role}[{', '.join(sorted(tools))}]" for role, tools in allowed.items())
    return ", ".join(sorted(allowed))

# 3단계: 쓰기 게이트는 run_case가 모든 구조의 마지막 답에 부릅니다. 그 소스에서 확인합니다.
gate_in_runner = "gate_answer(" in inspect.getsource(runner._run_turns)

harness_rows = []
for name in STRUCTURES:
    # 4단계: 구조마다 새 Desk로 만듭니다. 만들기만 하고 invoke하지 않습니다.
    harness_desk = make_desk(case_by_id["complex-double-pay"], label=f"harness/{name}")
    app = BUILDERS[name](harness_desk)
    source = inspect.getsource(BUILDER_SOURCE[name])

    # 5단계: 예산. 구조 소스가 desk.budget을 허용 목록 middleware에 넘기고, 한도가 FIXED_BUDGET과 같은지 봅니다.
    same_limits = all(getattr(harness_desk.budget, key) == value for key, value in FIXED_BUDGET.items())
    uses_desk_budget = "desk.budget" in source or "checked_worker(" in source
    budget_text = "예 (desk.budget, 한도 FIXED_BUDGET)" if same_limits and uses_desk_budget else "아니오"

    # 6단계: 최종 응답 형식. 소스의 ProviderStrategy(...) 안 스키마 이름을 읽습니다.
    formats = sorted(set(re.findall(r"ProviderStrategy\((\w+)\)", source)))

    # 7단계: 쓰기 게이트. hybrid는 그래프 안에도 gate 노드가 있습니다.
    gate_text = "예 (run_case의 gate_answer" + (" + gate 노드)" if name == "hybrid" else ")") if gate_in_runner else "아니오"

    # 8단계: 자동 general-purpose Worker. task 도구의 담당자 목록을 task_targets로 읽습니다.
    if name in GRAPH_STRUCTURES:
        probe = create_deep_agent(model=model, tools=harness_desk.tools("all"))  # 노드 안에서 만드는 것과 같은 설정
        targets, where = task_targets(probe), "같은 설정의 Agent"
    else:
        targets, where = task_targets(app), "구조 자체"
    auto_worker = "있음" if "general-purpose" in (targets or []) else "없음"

    harness_rows.append({"구조": name, "같은 Budget 객체를 쓰는가": budget_text, "도구 허용 목록": allow_text(ALLOWED[name]),
                         "최종 응답 형식": ", ".join(formats), "쓰기 게이트(gate_answer)": gate_text,
                         "자동 general-purpose Worker": f"{auto_worker} ({where}에서 확인, task 담당자 {targets})"})

show_table(harness_rows)
leaked = [row["구조"] for row in harness_rows if row["자동 general-purpose Worker"].startswith("있음")]
assert not leaked, f"자동 general-purpose Worker가 남은 구조: {leaked}"
print("모델 호출 0회")

**2026년 11월 주요 프레임워크 공식 문서가 기본으로 제공하는 멀티에이전트 구성과 이 노트북의 구조.**

> 어느 구성이 가장 많이 쓰이는지 재 본 조사는 찾지 못했습니다. 출처 없이 도는 70% 같은 수치는 인용하지 않습니다. 그래서 이 표는 많이 쓰이는 순서가 아니라 **공식 문서가 2026-11-02 기준으로 기본 구성으로 싣고 있는 것**을 이 노트북의 구조와 짝지은 것입니다. 사용 빈도에 대해서는 아무 말도 하지 않습니다.

| 이 노트북의 구조 | 절 | 공식 문서의 구성 |
|---|---|---|
| Supervisor (`build_supervisor`) | 02 3절 | LangChain Subagents, OpenAI Agents as tools (manager), Microsoft Magentic (manager agent), Google ADK Collaborative workflow (coordinator가 sub-agent를 부림), Anthropic orchestrator-worker (lead agent와 병렬 subagent) |
| Handoff (`build_handoff`) | 02 4절 | LangChain Handoffs, OpenAI Handoffs, Microsoft Handoff |
| Router + Workflow (`build_hybrid`) | 03 | LangChain Router와 Custom workflow, OpenAI code orchestration (structured output으로 routing, chaining, evaluator loop, parallel), Microsoft Sequential과 Concurrent, Google ADK Template workflows (sequence, loop, parallel)와 Routing |
| Skills (`single_skills`) | 이 노트북 4절 | LangChain Skills |
| 이 과정에서 다루지 않음 | | Microsoft Group Chat |

읽는 법은 세 가지입니다.

- 이름이 달라도 모양이 같은 것이 많습니다. "누가 다음 행동을 정하는가"와 "결과가 돌아오는가"로 보면 manager, orchestrator, coordinator는 모두 Supervisor 쪽이고, Handoff는 대화를 넘기는 쪽입니다.
- 코드가 순서를 정하는 구성(Sequential, Concurrent, Template workflows)은 이 노트북에서는 Workflow 쪽, 곧 hybrid의 그래프에 해당합니다. 0절에서 코드가 맡은 칸을 Agent로 나누지 않은 것과 같은 판단입니다.
- 표는 구성의 **이름**을 짝지을 뿐 성능을 비교하지 않습니다. 어느 쪽이 낫다는 말은 이 노트북의 같은 조건 비교표로 자기 사례에서 재야 합니다.

출처 (2026-11-02에 확인한 공식 문서와 글):

- LangChain Multi-agent: https://docs.langchain.com/oss/python/langchain/multi-agent
- OpenAI Agents SDK, Multi-agent orchestration: https://openai.github.io/openai-agents-python/multi_agent/
- Microsoft Agent Framework, Workflow orchestrations (문서 갱신일 2026-08-25): https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations
- Google ADK, Workflows: https://adk.dev/workflows/
- Anthropic, How we built our multi-agent research system (2025-06-13): https://www.anthropic.com/engineering/multi-agent-research-system

In [ ]:
CONFIG = {"STRUCTURES": ["single", "supervisor", "hybrid"], "CASE_TYPES": ["simple", "complex"], "REPEATS": 1}
grid_cases = [case for case in cases if case["type"] in CONFIG["CASE_TYPES"]]
runs = len(CONFIG["STRUCTURES"]) * len(grid_cases) * CONFIG["REPEATS"]
measured = {"single": single_double["model_calls"], "supervisor": supervisor_row["model_calls"], "hybrid": hybrid_rows[1]["model_calls"]}
worst = sum(measured[name] for name in CONFIG["STRUCTURES"]) * len(grid_cases) * CONFIG["REPEATS"]
show_table([{"고정 조건": key, "값": value} for key, value in {
    "모델": settings.model, "effort": settings.effort, "SDK 재시도": "없음(max_retries=0)", "도구": sorted(ALL_TOOLS), "검색": "BM25", "예산": FIXED_BUDGET,
    "쓰기 경로": "gate_answer", "바꾸는 변수": "구조"}.items()])
show_table([{key: case[key] for key in ("id", "type", "entity", "as_of", "question")} for case in grid_cases])
print(f"{len(CONFIG['STRUCTURES'])}구조 × {len(grid_cases)}사례 × {CONFIG['REPEATS']}회 = {runs}행")
print(f"앞서 잰 복합 사례 호출로 모든 행을 잡은 상한: 모델 약 {worst}회 {measured}. 단순 사례는 이보다 적습니다. 모델 호출 0회.")

In [ ]:
announce("같은 조건 비교표", model_calls=100)
grid_rows = run_grid(CONFIG["STRUCTURES"], grid_cases, model=model, repeats=CONFIG["REPEATS"])
print(f"{len(grid_rows)}행 완료. 실패한 행도 그대로 남습니다.")

In [ ]:
grid_checked = [evaluate_row(row, gold) for row in grid_rows]
grid_table = [{"행": index, "구조": row["structure"], "유형": row["type"], "사례": row["case_id"], "상태": row["status"], "outcome": row["outcome"],
               "자동 검사": check["자동 검사"], "실패 칸": check["실패 칸"], "누락 사실": check["누락 사실"], "금지 포함": check["금지 포함"],
               "호출": row["model_calls"], "도구": row["tool_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"], "겹침": row["max_worker_overlap_seconds"],
               "최종 응답자": row["final_speaker"], "원장": row["ledger_refunds"], "오류": row["errors"][:1]} for index, (row, check) in enumerate(zip(grid_rows, grid_checked, strict=True))]
show_table(grid_table)
print(f"전체 {len(grid_table)}행. 지운 행 없음. 자동 검사 실패 {sum(not row['자동 검사'] for row in grid_table)}행, 실행 실패 {sum(row['상태'] != 'ok' for row in grid_table)}행. 모델 호출 0회.")

In [ ]:
grid_summary = summarize_rows(grid_rows, grid_checked)
show_table(grid_summary, columns=["구조", "유형", "실행", "자동 검사 통과", "실행 실패", "평균 시간", "평균 모델 호출", "평균 입력 토큰", "usage 누락", "정책 위반"])
print(f"각 칸의 표본은 {CONFIG['REPEATS']}회 반복, 유형당 {len(grid_cases) // len(CONFIG['CASE_TYPES'])}사례입니다. 반복이 3회 미만이므로 통계 주장을 하지 않습니다. 모델 호출 0회.")

In [ ]:
twins = []
for name, earlier in (("single", single_rows[0]), ("supervisor", supervisor_row), ("hybrid", hybrid_rows[1])):
    later = next(row for row in grid_rows if row["structure"] == name and row["case_id"] == "complex-double-pay")
    for label, row in (("앞 절 실행", earlier), ("비교표 실행", later)):
        twins.append({"구조": name, "실행": label, "호출": row["model_calls"], "도구": row["tool_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"],
                      "자동 검사": evaluate_row(row, gold)["자동 검사"]})
show_table(twins)
print("같은 구조, 같은 사례, 같은 조건을 두 번 잰 값입니다. 추가 실행 없이 이미 있는 행을 나란히 놓았습니다. 모델 호출 0회.")

In [ ]:
if not RUN_OPTIONAL:
    print("실행하지 않았습니다. 첫 코드 셀의 RUN_OPTIONAL을 True로 바꾸세요. 모델 호출 약 220~330회, 10~15분 규모입니다.")
else:
    announce("전체 비교 4유형 × 5구조 × 1회", model_calls=330)
    full_cases = [case for case in cases if case["type"] in ("simple", "complex", "insufficient", "approval")]
    full_rows = run_grid(list(STRUCTURES), full_cases, model=model)
    full_checked = [evaluate_row(row, gold) for row in full_rows]
    show_table(summarize_rows(full_rows, full_checked))

In [ ]:
# 실패 재현: 평균만 보면 잘못 읽히는 표를 만듭니다. 빨리 끝난 실패 행을 예산을 낮춰 일부러 하나 만듭니다.
announce("예산을 낮춘 실패 행 한 개", model_calls=3)
cut_row = run_case("single", double_pay, model=model, budget={**FIXED_BUDGET, "max_model_calls": 2})
same = [pair for pair in zip(grid_rows, grid_checked, strict=True) if pair[0]["type"] == "complex"]
mixed = [*same, (cut_row, evaluate_row(cut_row, gold)), (forced_rows[0], evaluate_row(forced_rows[0], gold))]  # 예산을 낮춘 행과 Fast를 강제한 행을 섞습니다.
names = CONFIG["STRUCTURES"]

def means(pairs):
    count, tokens = len(pairs), [row["input_tokens"] for row, _ in pairs if row["input_tokens"] is not None]
    average = lambda key: round(sum(row[key] for row, _ in pairs) / count, 1) if count else None
    return {"행": count, "평균 시간": average("elapsed_seconds"), "평균 호출": average("model_calls"), "평균 입력 토큰": round(sum(tokens) / len(tokens)) if tokens else None,
            "자동 검사 통과": sum(check["자동 검사"] for _, check in pairs), "남은 사례": sorted({row["case_id"] for row, _ in pairs})}

def table(pairs, name_filter=lambda pair: True):
    return [{"구조": name, **means([p for p in pairs if p[0]["structure"] == name and name_filter(p)]),
             "제외한 행": sum(p[0]["structure"] == name and not name_filter(p) for p in pairs)} for name in names]

print("평균 0. 같은 조건의 복합 사례, 실패 행 포함"); show_table(table(same))
print("평균 A. 평균 0에 빨리 끝난 실패 행 2개를 섞음(실패 행 포함)"); show_table(table(mixed))
print("평균 B. 평균 A에서 자동 검사 실패 행을 뺌(실패 행 제외)"); show_table(table(mixed, lambda pair: pair[1]["자동 검사"]))
print("실패 행 목록. 평균 표와 무관하게 지우지 않습니다.")
show_table([{"구조": row["structure"], "사례": row["case_id"], "상태": row["status"], "호출": row["model_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"],
             "누락 사실": check["누락 사실"], "실패 칸": check["실패 칸"]} for row, check in mixed if not check["자동 검사"]])

In [ ]:
failed_rows = [row for row, check in zip(grid_rows, grid_checked, strict=True) if not check["자동 검사"]]  # 자동 검사 실패 행을 먼저 사람이 봅니다.
review_row = failed_rows[0] if failed_rows else next(row for row in grid_rows if row["case_id"] == "complex-double-pay")
print(f"[답] 구조={review_row['structure']} 사례={review_row['case_id']} outcome={review_row['outcome']}\n{review_row['answer']}\n")
print("[자동 검사]", {key: value for key, value in evaluate_row(review_row, gold).items() if key in ("자동 검사", "실패 칸", "누락 사실", "금지 포함")})
print("[인용]", review_row["evidence_refs"])
documents = build_retriever().documents
for doc_id in sorted({*review_row["cited_docs"], *gold[review_row["case_id"]]["required_docs"]}):
    print(f"\n[원문] {documents[doc_id].source}")
    for number, line in enumerate(documents[doc_id].text.split("\n"), start=1):
        if line.strip() and any(ch.isdigit() for ch in line):
            print(f"  {number}: {line}")
# 답의 문장과 원문 행을 대조해 자동 검사와 다르게 볼 곳을 적으세요.
HUMAN_REVIEW = {"대조한 문장": "미판정", "자동 검사와 다른 판단": "미판정", "근거 원문 행": "미판정"}
print("\nHUMAN_REVIEW:", HUMAN_REVIEW, "| 모델 호출 0회")

**관찰할 것.** 전체 행 표, 요약 표, 같은 구조로 두 번 잰 표, 평균 표 세 개, 사람 판정 칸을 차례로 읽으세요. **이 표는 어느 구조가 이겼는지 말하지 않습니다.**

**1회 관찰.** 요약 표의 각 칸은 유형당 사례 몇 개, 사례당 1회입니다. 반복 없이 비교했으니 구조의 차이와 실행마다의 변동을 구분할 수 없습니다. 같은 구조로 두 번 잰 표를 보세요. 같은 구조, 같은 사례인데 `호출`, `입력 토큰`, `자동 검사`가 얼마나 달라졌나요? 구조 사이의 차이가 이 변동보다 작다면 그 차이는 구조 때문이라고 말할 수 없습니다. 말할 수 있는 것은 "이번 실행에서 이 조합의 이 행이 이렇게 나왔다"입니다.

**평균의 함정.** 세 평균 표를 차례로 비교하세요.

- **평균 0에서 A로.** 사례를 바꾸지 않고 빨리 끝난 실패 행만 더했습니다. `평균 시간`과 `평균 입력 토큰`이 내려갔다면 **실패가 평균을 좋아 보이게 만든 것**입니다. `행` 대비 `자동 검사 통과`가 어떻게 변했는지 함께 보세요. 평균만 내면 그 구조가 비용을 줄인 것처럼 읽힙니다.
- **A에서 B로.** 실패 행을 빼면 `남은 사례`가 달라질 수 있습니다. 어려운 사례가 빠져 쉬운 사례만 남으면 평균이 내려가고, 반대로 어려운 사례만 남으면 평균이 올라갑니다. 빠진 행이 빨리 끝난 실패 행이라면 평균이 올라가기도 하니, 움직인 방향보다 `남은 사례`가 달라졌는지를 먼저 보세요. 이때는 구조마다 **다른 사례**의 평균이 되므로 비교가 되지 않습니다. `남은 사례`가 그대로라면 B는 평균 0으로 돌아간 것이고, 빼낸 실패 행이 왜 났는지는 여전히 표에서 사라져 있습니다. `제외한 행`이 있는 구조의 B 값을 그대로 읽지 마세요.
- **실패 행 목록.** `상태`가 ok인 실패 행은 실행은 끝났지만 자동 검사를 통과하지 못한 행입니다. `누락 사실`과 `실패 칸`을 답과 대조하세요. 정말 빠진 행과 표현만 다른 행은 다르게 다뤄야 합니다. `누락 사실`은 비었는데 `실패 칸`에 `outcome`이나 `인용 문서`가 있다면 문자열은 답에 있어도 기대한 결과나 근거 연결이 어긋난 행입니다. 전체 행 표의 `outcome`과 답을 읽고 어느 쪽인지 가르세요. `outcome`만 어긋났다면 요청하지 않은 환불을 제안한 것처럼 답이 다른 종류의 결과로 끝난 것이고, `insufficient`로 끝나면서 `인용 문서`도 어긋났다면 Worker의 결과가 근거 참조 없이 넘어왔을 가능성을 살펴보세요. 그래서 평균 표는 실패 행 목록과 항상 함께 읽습니다.

**usage 누락.** 요약 표의 `usage 누락`이 0이 아니면 그만큼 토큰 평균이 실제보다 낮습니다. 0으로 세지 않고 빼서 계산한 값입니다.

**자동 검사의 한계.** 사람 판정 칸은 자동 검사가 실패한 행을 먼저 보여 줍니다(없으면 복합 사례 행을 보여 줍니다). 답과 원문을 나란히 보세요. 자동 검사는 표현이 다르면 맞는 답을 놓치고, 문자열이 있으면 틀린 맥락도 통과시킵니다. 통과 행에서도 인용이 그 문장을 실제로 뒷받침하는지 대조하세요. 금액이 맞아도 기준 조항을 잘못 인용했을 수 있습니다. 판정이 자동 검사와 다르면 `HUMAN_REVIEW`에 적으세요. 전체 행 표의 `자동 검사` 숫자를 사람 판정으로 고쳐 읽으면 요약 표의 인상이 어떻게 바뀌는지도 확인해 보세요.

**결론을 말하려면 무엇을 더 재야 할까요?** 반복 횟수를 늘려 변동을 보고, 사례 수와 유형을 늘리고, 답을 사람이 판정한 칸을 더해야 합니다. 그래도 이 수업의 사례와 모델 안에서만 말할 수 있습니다. 또 Router의 오판 비율, 실제 트래픽의 유형 구성, 호출당 가격은 이 표에 없습니다. 이 표만으로 운영에서 어떤 구조가 더 낫다고 말할 수 없다는 점을 기억하세요.

## 8. 선택 기록 (3분)

**문제.** Agent 수와 조직도는 결과물이 아닙니다. 다음 사람이 구조를 바꿀지 판단할 때 필요한 것은 **왜 이 구조를 골랐고, 무엇을 포기했으며, 어떤 결과가 나오면 다시 생각할지**입니다. 설계 이유가 결과물입니다.

선택할 때 답할 설계 질문은 여섯 가지입니다.

1. **업무 목표.** 어떤 상태를 완료로 볼까요? (01)
2. **Agent·SubAgent의 필요성.** 코드나 Skill로 충분하지 않은 이유는 무엇인가요? (02, 이 노트북 4절)
3. **작업 Graph.** 순차, 병렬, 분기, 루프는 어디에 있나요? (03)
4. **Context.** 각 작업이 보는 것과 안 보는 것은 무엇인가요? (04)
5. **권한과 예산, 종료.** 누가 쓸 수 있고, 어디서 멈추나요? (04)
6. **평가 근거.** 어떤 결과로 이 선택을 정당화하나요? (이 노트북 7절)

아래 `DECISION`에 사례 유형별로 고른 구조, 근거가 되는 비교표의 `행` 번호, 포기한 것을 적으세요. 위 표를 보고 "미판정"으로 남겨도 됩니다. 1회 관찰로 고를 수 없다면 그 사실과 무엇을 더 재야 할지를 적는 것도 기록입니다.

In [ ]:
DECISION = {"simple": {"구조": "미판정", "근거 행": [], "포기한 것": "미판정"},
            "complex": {"구조": "미판정", "근거 행": [], "포기한 것": "미판정"}}
# DECISION["simple"] = {"구조": "...", "근거 행": [0, 1], "포기한 것": "예: Worker를 나눠 얻는 원문 격리"}

show_table([{"유형": kind, **entry, "근거 행 내용": [(grid_table[i]["구조"], grid_table[i]["사례"], grid_table[i]["자동 검사"]) for i in entry["근거 행"]]}
            for kind, entry in DECISION.items()])
print("근거 행 번호는 7절 전체 행 표의 '행' 칸입니다. 모델 호출 0회.")

## 직접 해보기

유료 셀은 첫 줄의 `RUN_EXERCISE = False`를 `True`로 바꿔야 돌아갑니다. 수강생이 바꿀 값은 그 아래 대문자 상수로 모아 두었습니다. 모델을 부르지 않는 셀은 그대로 실행됩니다.

### 해보기 1

(실제 호출) 한 사례를 3회 반복해 호출 수와 답의 변동을 봅니다. 앞서 본 구조 사이의 차이가 이 변동보다 큰지 비교해 보세요. 반복이 3회여도 통계 주장에는 부족하다는 점을 잊지 마세요.

In [ ]:
RUN_EXERCISE = False
STRUCTURE, CASE_ID, REPEATS = "hybrid", "complex-double-pay", 3
if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("한 사례 3회 반복", model_calls=48)
    repeat_rows = run_grid([STRUCTURE], [case_by_id[CASE_ID]], model=model, repeats=REPEATS)
    show_table([{"반복": row["repeat"], "상태": row["status"], "호출": row["model_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"],
                 "자동 검사": evaluate_row(row, gold)["자동 검사"], "누락 사실": evaluate_row(row, gold)["누락 사실"], "답 앞부분": row["answer"][:60]} for row in repeat_rows])

### 해보기 2

예산을 **낮춰** 어느 구조가 먼저 실패하고 사유가 무엇인지 봅니다. 올려서 실패를 감추는 실험이 아닙니다. 값을 올리면 아래 `assert`가 멈춥니다. 실행 전에 규모가 먼저 출력되므로 보고 나서 켤지 정하세요.

In [ ]:
RUN_EXERCISE = False
LOWER_BUDGET = {**FIXED_BUDGET, "max_model_calls": 6, "max_seconds": 60}
TARGETS = ["single", "supervisor", "hybrid"]
assert all(LOWER_BUDGET[key] <= FIXED_BUDGET[key] for key in FIXED_BUDGET), "예산을 올려서 실패를 감추지 않습니다."
print("낮춘 예산:", LOWER_BUDGET, "| 원래 예산:", FIXED_BUDGET)
print(f"켜면 {len(TARGETS)}구조 × 1사례, 모델 호출 약 {len(TARGETS) * LOWER_BUDGET['max_model_calls']}회입니다. Router 호출도 예산이 셉니다.")
if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("낮춘 예산으로 3구조", model_calls=24)
    lowered = [run_case(name, double_pay, model=model, budget=LOWER_BUDGET) for name in TARGETS]
    show_table([{"구조": row["structure"], "상태": row["status"], "호출": row["model_calls"], "시간": row["elapsed_seconds"], "사유": row["errors"][:1], "outcome": row["outcome"]} for row in lowered])

### 해보기 3

(실제 호출) `billing-policy`의 표현을 바꾼 새 사례를 직접 더합니다. 기대 사실은 반드시 **원문 행**에 연결하세요. 아래 셀은 기대 표현이 들어 있는 원문 행을 먼저 출력합니다. 사례 파일은 고치지 않고 이 셀의 변수로만 평가합니다.

In [ ]:
RUN_EXERCISE = False
NEW_CASE = {"id": "my-refund-window", "type": "simple", "entity": "알파", "as_of": "2026-09-08", "question": "결제하고 석 달쯤 지나면 환불은 어떻게 처리되나요?"}
NEW_EXPECT = {"outcome": "answered", "required_any": [["90일"], ["재무 담당", "심사"]], "required_docs": ["billing-policy"], "forbidden": [], "ledger_refunds": 0, "route": "fast"}
policy_text = build_retriever().documents["billing-policy"].text.split("\n")
print("기대 사실의 원문 행:", [(number, line) for number, line in enumerate(policy_text, start=1) if "90일" in line])
if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("새 사례 한 건", model_calls=15)
    new_row = run_case("single", NEW_CASE, model=model)
    show_table([evaluate_row(new_row, {**gold, NEW_CASE["id"]: NEW_EXPECT})], columns=["case_id", "자동 검사", "실제 outcome", "누락 사실", "누락 문서", "실패 칸"])
    print(new_row["answer"])

### 해보기 4

자동 검사의 `required_any`로는 틀렸다고 판정되는 **올바른 답**, 또는 그 반대로 통과하는 **틀린 답**을 하나 찾아 이유를 설명합니다. 모델을 부르지 않습니다. 아래 두 답은 예시이니 자기 답으로 바꿔 보세요.

In [ ]:
BASE = {"structure": "손으로 쓴 답", "case_id": "complex-double-pay", "status": "ok", "outcome": "needs_approval", "ledger_refunds": 0,
        "cited_docs": ["billing-policy", "sla-a-v2", "sla-a-amendment"]}
CORRECT_BUT_FAILS = "P-9002 1,200,000원을 승인 후 환불합니다. 가용률 99.3%로 크레딧은 기본 이용료의 15%, 15만 원이고 2026-09-20까지 신청하세요."
WRONG_BUT_PASSES = "P-9002는 환불 대상이 아닙니다. 1,200,000원은 그대로 둡니다. 99.3%라도 15%를 받지 못합니다. 150,000원은 드리지 않으며 9월 20일은 무관합니다."
show_table([{"답": answer, **{key: check[key] for key in ("자동 검사", "누락 사실", "금지 포함")}}
            for answer in (CORRECT_BUT_FAILS, WRONG_BUT_PASSES) for check in [evaluate_row({**BASE, "answer": answer}, gold)]])
print("표현이 다르면 맞는 답을 놓치고, 문맥을 모르므로 뜻이 반대여도 통과합니다. 모델 호출 0회.")

In [ ]:
EVALUATION_BASIS = "미작성: 어떤 결과로 구조 선택을 정당화할지, 무엇을 더 재야 하는지 적으세요."
answer = json.dumps({"근거": EVALUATION_BASIS, "선택": DECISION, "1절 진단": DIAGNOSIS, "사람 판정": HUMAN_REVIEW}, ensure_ascii=False)
path = write_design_log("평가 근거: 어떤 결과로 선택을 정당화할까?", answer)
print("기록 위치:", path.relative_to(ROOT))
print("기록된 질문:", list(read_design_log()))

## 정리

- **순서가 먼저입니다.** Single의 실패를 네 칸으로 분류하고, 프롬프트·도구·검색으로 고칠 수 있는 것을 먼저 고칩니다. 분리의 근거는 고친 뒤에도 남는 실패나 비용입니다.
- **이득과 비용은 같은 행에 씁니다.** 호출, 토큰, 시간, 자동 검사, 조정 바이트를 함께 읽습니다. 품질이 같다면 추가 비용이 무엇을 샀는지 답해야 합니다.
- **나누는 기준은 Persona가 아니라 Context 경계입니다.** 역할 이름은 도구 호출을 막지 못합니다. Context 분리 역시 보안 격리가 아니므로 권한은 따로 설계합니다.
- **절차는 Skill, 별도 실행 Context는 SubAgent, 판단이 없는 것은 코드**입니다. 조합할 수 있습니다.
- **수치는 네 칸(비교 대상, 평가 방법, 표본, 측정자)과 함께 말합니다.** 지연은 Agent 수가 아니라 가장 긴 경로로 읽습니다.
- **Fast Path는 검사를 생략하지 않습니다.** 이득은 오판과 복구 비용까지 합쳐서 말합니다. 빠진 범위는 구조 검증이 잡지 못할 수 있습니다.
- **실패 행을 지우지 않습니다.** 실패가 빨리 끝나면 평균이 좋아 보입니다. 실패와 통과를 함께 읽습니다.
- **1회 관찰로 구조의 우열을 말하지 않습니다.** 말할 수 없는 것을 적는 것까지가 비교표의 결과입니다.

## 확인 질문

1. Multi-Agent로 바꾸기 전에 고쳐야 할 실패와 분리의 근거가 되는 실패를 자기 표에서 하나씩 들어 보세요.
2. 조정 비용은 어느 칸에 나타나나요? 품질이 같을 때 무엇을 근거로 비용을 정당화하나요?
3. "Context 분리는 곧 격리"라는 주장이 틀린 이유를 3절의 `같은 Desk`, `같은 원장` 출력으로 설명해 보세요.
4. 비교 대상이 다른 두 수치를 합쳐 말하면 왜 틀리나요? 자기 표에서 예를 들어 보세요.
5. 평균 지연이 낮아진 구조가 나빠진 경우는 어떤 경우인가요? 평균 0, 평균 A, 실패 행 목록으로 설명해 보세요.
6. 이 표만으로 말할 수 없는 것을 세 가지 적어 보세요.

**다음 노트북.** 06에서는 EXA로 병렬, 직렬, 연쇄 리서치 Deep Agent를 만들어 같은 주제로 비교합니다. 고객 요청이 아니라 처음 보는 주제에서, 이 노트북의 판단 방식을 직접 적용합니다.